# afMLevel Video Demonstration Notebook

This notebook provides an introduction to using the **afMLevel** package and its trained U‑Net models to level Atomic Force Microscopy (AFM) image stacks or high-speed AFM videos.

To understand the basics of how the **afMLevel** package and the AFM Machine Learning Levelling methods work, open and run through the `afMLevel_demo.ipynb` notebook to see a detailed demonstration of the software on single image frames. In this notebook **afMLevel** is combined with the [playNano](https://github.com/derollins/playNano/) package to load and run the ML levelling models on time-series or high-speed AFM data.

To explore how the models work, simply work through this notebook by clicking on each code cell and running it (using **Shift** + **Enter**) to view the output. You can experiment with different parameters by editing the code, and re‑running a cell will immediately show how your changes affect the results. You can also switch from the provided demonstration data to your own AFM datasets by adjusting the file paths. Notebooks are designed for exploration, so you can rerun cells or restart the kernel at any time without worrying about breaking anything.

## 1. Setup and Imports

Start by ensuring all the required packages and functions are installed and imported. The two trained U-Net models are downloaded if not already available and the `lutAFM` AFM image colourmap loaded.

If you do not have **afMLevel** and **playNano** already installed, uncomment the relevant lines in the next block to install the required packages. 

In [ ]:

#Ignore this cell if you have installed afMLevel with the `pip install -e .[notebooks]` command, all the notebook dependancies will already be installed.

# If afMLevel isn't installed ensure this notebook is opened from the
# repository root and uncomment the installation line (starting !).
# Install in editable mode if developing:
# !pip install -e .

# To install the playNano package uncomment the following line and run
# to install playNano.
# !pip install playnano


Now import the required packages, download the trained models if required and load the AFM colourmap.

In [ ]:

from pathlib import Path
import os
import requests
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib import animation

import numpy as np

from playnano.afm_stack import AFMImageStack
from playnano.processing.pipeline import ProcessingPipeline

from afmlevel.background_model import level_ml_bg
from afmlevel.mask_model import level_ml_mask, ml_mask, ml_edges

# Load AFM colourmap
AFM = np.load('./lutAFM.npy') # by default the working directory is the folder the notebook is in, i.e. /notebooks
AFM = ListedColormap(AFM)


**playNano** is a package that allows the frames of an AFM video to be processed together. **afMLevel** is available as a plugin for **playNano** to allow the easy, traceable and quick processing of AFM videos with the ML levelling methods.

Example data can be downloaded for the demonstration, alternatively this can be skipped, and you can set the `data_path` variable in the subsequent cell to point to your own data and run that. 

The example data is from Heath, G. R. (2026). Annexin V assembly dynamics on lipid bilayers by High-Speed AFM (Version 1) [Data set]. Zenodo. https://doi.org/10.5281/zenodo.19254504 .   

In [ ]:

# Download example data from this dataset: https://zenodo.org/records/19254504

import zipfile

# Define URLs and paths
zip_url = "https://zenodo.org/records/19254504/files/Annexin%20V%20HS%20AFM.zip?download=1"
local_zip = Path("data/dataset_video.zip")
extract_dir = Path("data/extracted_video")

extract_dir.mkdir(parents=True, exist_ok=True)

# Download the zip file
if not local_zip.exists():
    print("Downloading dataset...")
    r = requests.get(zip_url, stream=True)
    r.raise_for_status()

    with open(local_zip, "wb") as f:
        for chunk in r.iter_content(chunk_size=8192):
            f.write(chunk)
else:
    print("Using cached download:", local_zip)

# Unzip
print("Extracting...")
with zipfile.ZipFile(local_zip, "r") as z:
    z.extractall(extract_dir)

print("Done. Files extracted to:", extract_dir)


To begin working with AFM videos we need to load the data into a **playNano** `AFMIMageStack` object. If you would like to process your own data edit the `data_path` variable to point to your folder of `.jpk` or `.spm` images or `.asd` or `.h5-jpk` file and update the channel variable to whichever channel you would like to process.

In [ ]:

data_path = Path("data/extracted_video/For Zenodo/imaging-14.39.16.156/") # edit this to use your own data oe explore other videos in the folder.

channel = 'height_retrace'  # common HS-AFM height channel

stack = AFMImageStack.load_data(data_path, channel=channel)

print(f"Loaded {stack.n_frames} frames; each frame shape: {stack.image_shape}")
print(f"Pixel size: {stack.pixel_size_nm} nm")

fig, ax = plt.subplots(figsize=(4,4))
im = ax.imshow(stack.data[0], cmap=AFM, origin='lower')
cb = plt.colorbar(im, label="Height / nm")
plt.title("Raw Frame 0")
ax.axis('off')


To process data in **playNano** a processing pipeline is built from available processing steps and then run on a `AFMIMageStack` object. If both **afMLevel** and  **playNano** are installed then the `level_ml_bg()` and `level_ml_mask()` functions are added as available steps in **playNano**, `level_ml_bg` and `level_ml_mask` respectively. Using the `ProcessingPipeline.add_filter` method, `level_ml_bg` requires the model_path parameter (set to the location of the trained background U- Net model file). The `level_ml_mask` step require a `method` (set to `"iterative ML mask"` by default) and the `model_path` should point to the trained mask model. 

### Background Model Demo

In [ ]:

# Restore raw data
# Restore raw data if available
raw_data = stack.processed.get('raw')
if raw_data is not None:
    stack.data = raw_data

pipe = ProcessingPipeline(stack)

# Apply the background level routine three times

pipe.add_filter('drop_frames', indices_to_drop = [6]) # remove final frame 
pipe.add_filter('level_ml_bg')
pipe.add_filter('level_ml_bg')
pipe.add_filter('level_ml_bg')

proc_record = pipe.run()

# The processing steps and the associated parameters are stored
# within the 'providence' AFMImageStack attribute.
# Accessing this provides are record of the processing steps used.
print("Processing steps executed with parameters:")
for step in stack.provenance['processing']['steps']:
    print(f" • {step['name']}: {step['params']}")


fig, ax = plt.subplots(figsize=(4,4))
im = ax.imshow(stack.data[0], cmap=AFM, origin='lower')
cb = plt.colorbar(im, label="Height / nm")
plt.title("Processed Frame 0")
ax.axis('off')


In [ ]:

from IPython.display import HTML

def update(frame):
    global cb
    if cb:
        cb.remove()
        cb = None
    im.set_data(stack.data[frame])
    ax.set_title(f"Frame {frame}")
    return [im]


plt.rcParams['animation.embed_limit'] = 100
anim = animation.FuncAnimation(fig, update, frames=range(stack.n_frames), interval=100)
HTML(anim.to_jshtml())


### Mask Model Demo

Load a second dataset and processes this with the mask model within the "iterative-ml-mask" method.

In [ ]:

data_path_mask = Path("data/extracted_video/For Zenodo/imaging-14.01.11.254") # edit this to use your own data oe explore other videos in the folder.

channel = 'height_retrace'  # common HS-AFM height channel

stack_mask = AFMImageStack.load_data(data_path_mask, channel=channel)

print(f"Loaded {stack_mask.n_frames} frames; each frame shape: {stack_mask.image_shape}")
print(f"Pixel size: {stack_mask.pixel_size_nm} nm")

fig, ax = plt.subplots(figsize=(4,4))
im_mask = ax.imshow(stack_mask.data[0], cmap=AFM, origin='lower')
cb = plt.colorbar(im_mask, label="Height / nm")
plt.title("Raw Frame 0")
ax.axis('off')


In [ ]:

# Restore raw data
# Restore raw data if available
raw_data_mask = stack_mask.processed.get('raw')
if raw_data_mask is not None:
    stack_mask.data = raw_data_mask

pipe_mask = ProcessingPipeline(stack_mask)

#Level with the mask model then set the mean value of the protein layer to 0.

pipe_mask.add_filter('drop_frames', indices_to_drop = [15]) # remove final frame 
pipe_mask.add_filter('level_ml_mask', method="iterative-ml-mask")
pipe_mask.add_mask('mask_below_threshold', threshold=0)
pipe_mask.add_filter('zero_mean')

proc_record_mask = pipe_mask.run()

# The processing steps and the associated parameters are stored
# within the 'providence' AFMImageStack attribute.
# Accessing this provides are record of the processing steps used.
print("Processing steps executed with parameters:")
for step in stack_mask.provenance['processing']['steps']:
    print(f" • {step['name']}: {step['params']}")


fig, ax = plt.subplots(figsize=(4,4))
im_mask = ax.imshow(stack_mask.data[0], cmap=AFM, origin='lower')
cb = plt.colorbar(im_mask, label="Height / nm")
plt.title("Processed Frame 0")
ax.axis('off')


In [ ]:

from IPython.display import HTML, Image

def update(frame):
    global cb
    if cb:
        cb.remove()
        cb = None
    im_mask.set_data(stack_mask.data[frame])
    ax.set_title(f"Frame {frame}")
    return [im]


plt.rcParams['animation.embed_limit'] = 100
anim = animation.FuncAnimation(fig, update, frames=range(stack_mask.n_frames), interval=100)
HTML(anim.to_jshtml())


## Exporting GIFs

You may export the processed data as an animated GIF using the **playNano** `export_gif()` function. Run the following cells to output and display the GIFs.

In [ ]:

from playnano.io.gif_export import export_gif

# Set the output folder path and name
output_path = Path("data/output")
output_name="background_export"

# Create the directory if it doesn't exist
output_path.mkdir(parents=True, exist_ok=True)

# Export the background GIF
export_gif(
    stack,
    make_gif=True,
    output_folder=output_path,
    output_name=output_name,
    scale_bar_nm = 20,
    zmin=-1,
    zmax=1
)
full_output_path = f"./{output_path}/{output_name}_filtered.gif"

display(Image(filename=full_output_path))


In [ ]:

output_name_mask="mask_export"

# Export the mask GIF
export_gif(
    stack_mask,
    make_gif=True,
    output_folder=output_path,
    output_name=output_name_mask,
    scale_bar_nm = 50,
    zmin=-2,
    zmax=1,
)

full_output_path_mask = f"./{output_path}/{output_name_mask}_filtered.gif"

display(Image(filename=full_output_path_mask))
